In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and configures the environment.
import os, sys

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in instructor_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/instructor_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"✓ Setup complete — {os.getcwd()}")

# Module 1 — Welcome & Orientation
**Type:** [Watch Only]  
**Time:** 5 minutes  
**Job:** Set the rules, show the destination, establish the workflow pattern every module will follow.

---

> **Instructor note:** Do not rush this. Five minutes feels short but the rules set here — especially the Green/Red Path framing and the pooled scoring standard — will be referenced in every module. Students who miss this framing struggle later.

What I want to do today is keep this really practical. This is not meant to be one of those forecasting sessions where we build a model, get a number, and then all pretend that means we’re done.
In the real world, that’s not enough.
A forecast has to earn its keep. It has to beat something simple. It has to be reliable. And ideally, it has to be worth the extra complexity you’re introducing.
And honestly, that question matters even more now, because the bar has moved. It’s not just, “Can I beat Naive?” Now it’s also, “Can I beat a zero-shot foundation model without creating a bunch of extra work for myself?”
So that’s the lens we’re going to use today.
We’re going to walk through a full workflow using the Nixtla ecosystem — start with baselines, move into ML, look at a neural model, talk about prediction intervals, and at the end make an actual decision about what we’d feel good shipping.
So the goal here is not just to run some notebooks. The goal is to think a little more like, “If I had to own this in production, what would I actually trust?”
That’s really the mindset for the session.

---
## 1.1 — What We Are Building
**[Watch Only]**

> **Instructor note (1 min):** Walk through this list verbally. Do not just read it. For each item, say one sentence about why it matters operationally — not just what it is.

By the end of this session, you will have built a complete, scored forecasting pipeline on a real retail demand dataset.

Here is the destination:

| Module | What Gets Built | Artifact |
|---|---|---|
| 2 | Config locked, evaluation logic defined | `02_global_config.pkl` |
| 3 | Panel validated, data health confirmed | `03_validated_panel.parquet` |
| 4 | Naive, SeasonalNaive, AutoETS + Chronos benchmark | `04_baseline_forecasts.parquet` |
| 5 | LightGBM with lag and date features | `05_ml_forecasts.parquet` |
| 6 | NHITS global deep learning model | `06_dl_forecasts.parquet` |
| 7 | 80% intervals scored across all models | `07_uncertainty_leaderboard.parquet` |
| 8 | Final leaderboard, model selection decision | `08_final_master_leaderboard.csv` |

Every artifact is a checkpoint. If anything fails, you load the precomputed version and continue. **You will not fall behind.**

---
## 1.2 — The Dataset
**[Watch Only]**

> **Instructor note (1 min):** One sentence on M5, one sentence on why this specific subset was chosen, one sentence on what the evaluation window looks like. Keep it tight — EDA is Module 3.

**Dataset:** M5 Forecasting Competition (Walmart retail sales, Kaggle 2020)

**Our subset:** 1,000 item-store series. Selected by total sales volume, filtered for history depth and intermittency. Deterministic — `RANDOM_SEED = 42`, same result every time.

**Evaluation window:** 3 cross-validation cutoffs × 28-day horizon. Every metric is computed pooled across all windows and all series — not averaged per series first.

This is daily retail demand data. The patterns here — weekly seasonality, promotional spikes, intermittent SKUs — are the patterns that break naive approaches in production.

**NOTES**
Not saying short history or intermittent skus not important,  They are real - but not a good place for a general purpose workshop.  
Intermittent forecasting could get a dedicated workshop entirely on its own.
Different forecasting problem
Often for those simple baselines dominate - and then the conversation will be more a lesson on intermittency than on modern forecasting.

Short history also weaken the fairness of CV, and usable training content. 


---
## 1.3 — The Two Paths
**[Watch Only]**

> **Instructor note (1 min):** Show this section on screen and say it out loud. Students need to hear "you will not get stranded" before the session starts — it removes anxiety and keeps people from freezing when a cell is slow.

Every module has two routes.

**🟢 Green Path — live execution.** You run the cells. This is the goal.

**🔴 Red Path — checkpoint recovery.** If a cell fails or takes too long, you run one recovery cell and continue. Recovery takes under 2 seconds.

Red Path cells look like this in every notebook:

In [ ]:
# 🔴 RED PATH — this cell is shown for orientation only. Do not run it now.
# When a Green Path cell fails or exceeds the runtime ceiling,
# run the Red Path cell immediately below it.

from src.checkpointing import load_checkpoint
forecasts_df = load_checkpoint("04_baseline_forecasts")  # example only

**Runtime ceiling:** Any live cell must finish in under 90 seconds. If it does not, interrupt it (`I, I` in Jupyter / square stop button in Colab) and use the Red Path.

This is not a failure state. The Red Path exists because production-grade models on real data do not always fit in workshop time slots. **You are still running the full pipeline — you are just loading a stage instead of computing it.**

**NOTES**
- Green Path = intended live workflow

- Red Path has two jobs:
-- rescue people if a prior step breaks or they fall behind
-- keep the cohort moving when a full run would be too slow / too heavy / too unstable live
-- goal is to protect the learning flow, not make everyone wait on compute

key message:
Green Path = learn the mechanics
Red Path = protect the clock and keep everyone with the room


---
## 1.4 — Scoring Standard
**[Watch Only]**

> **Instructor note (1 min):** This is the most important slide of the session. The pooled scoring standard is what separates a real evaluation from a leaderboard contest. Say it slowly. Students who understand this will ask better questions in Module 8.

We judge every model on three metrics.

**wMAPE (Weighted Mean Absolute Percentage Error)**  
Measures point forecast accuracy. Pooled across all series and all backtest windows — not averaged per series first. This matters: per-series averaging gives equal weight to a SKU selling 1 unit/day and one selling 10,000. Pooling weights by volume.

**Bias**  
Systematic direction of error. Pooled across all series and windows: sum(ŷ − y) / sum(y). Positive = over-forecast. Negative = under-forecast. A model with good wMAPE but large bias is making consistently directional errors — dangerous in inventory planning.

**Interval Score (Winkler Score) at 80%**  
Measures uncertainty quality. An interval that is technically correct but spans "10 to 500 units" is useless to a procurement team. Interval Score penalizes width and misses together. Lower is better.

**Coverage (diagnostic only)**  
What fraction of actuals fell inside the 80% interval. We track it, but we do not rank on it — a model can hit 80% coverage with a dangerously wide interval.

The final question in Module 8 is not "which model has the best wMAPE?" It is: **"which model would you actually put into production, and why?"** Those are different questions.

**NOTES**
- Our book covers a lot of metrics — over 18 point forecast metrics — so the question is not **what exists**, it is **what are we optimizing for here?**
- We chose **wMAPE** because it lines up with how businesses usually feel forecast error
- **Pooled wMAPE** gives more weight to high-volume SKUs
- That is often a **feature, not a bug** — missing badly on a 10,000-unit SKU matters more than missing on a 1-unit SKU
- The tradeoff:
  - yes, a few high-volume categories can dominate
  - that is why in the real world you do **not** stop at one top-line score
  - you also review performance at multiple levels: portfolio, business unit, category, region
- Good phrase:
  - **wMAPE tells you how the business feels the error**

- On **Bias**:
  - this is underrated and often under-taught
  - accuracy tells you **how big** the errors are
  - bias tells you **which direction** the model is leaning
  - persistent positive bias = over-forecasting = excess inventory / working capital drag
  - persistent negative bias = under-forecasting = stockouts / service pain
- Good phrase:
  - **a model can look accurate on average and still be operationally dangerous if the bias is persistent**

- On **Interval Score / Winkler Score**:
  - easy interpretation: it rewards intervals that are **narrow and still correct**
  - if the interval misses the actual, it gets penalized
  - if the interval is absurdly wide, it also gets penalized
  - that is why it is better than just checking coverage alone

- On **Coverage**:
  - coverage is basically an honesty check
  - interval score is the usefulness check
  - a model can hit 80% coverage with intervals so wide that nobody can use them

- Closing thought:
  - we are not trying to find the prettiest leaderboard
  - we are trying to decide **what we would actually trust in production**

- Short sound bites:
  - **wMAPE tells you how the portfolio feels the miss**
  - **bias is direction, not just magnitude**
  - **persistent bias is usually more dangerous than people realize**
  - **coverage checks calibration; interval score checks usefulness**
  - **best metric score and best production decision are not always the same**

---
## 1.5 — Notebook Conventions
**[Watch Only]**

> **Instructor note (30 sec):** Quick visual scan — point at the labels on screen so students can pattern-match them in later notebooks.

Every section header in every notebook is labeled with one of three types:

| Label | Meaning |
|---|---|
| **[Watch Only]** | Instructor runs or explains. You watch. |
| **[Code With Me]** | You type. Active learning. |
| **[Take-Home Extension]** | Not covered live. In `take_home_bonus.ipynb`. |

Every major section also shows an **Expected Output** block before moving on. If your output does not match, stop and check — do not carry a bad state into the next module.

**[Code With Me]** sections in student notebooks have `__FILL_IN__` placeholders where the key lines go. One or two lines maximum. The surrounding code is always complete — you are filling in the mechanism, not rewriting the section.

---
## 1.6 — The Final Leaderboard (Destination Preview)
**[Watch Only]**

> **Instructor note (30 sec):** Show this table. Tell them: "This is where we are going. Every number in this table is something you will compute today. We will revisit it in Module 8 and make a real recommendation."
>
> The values below are illustrative. Do not read them as ground truth — actual results depend on the locked subset and CV windows.

This is what the final leaderboard looks like at the end of Module 8.

| Rank | Model | wMAPE | Bias | Interval Score (80%) | Coverage (80%) |
|---|---|---|---|---|---|
| 1 | LightGBM | — | — | — | — |
| 2 | NHITS | — | — | — | — |
| 3 | AutoETS | — | — | — | — |
| 4 | Chronos-t5-mini | — | — | — | — |
| 5 | SeasonalNaive | — | — | — | — |
| 6 | Naive | — | — | — | — |

Ranking order is illustrative. **The actual results may surprise you.** That is the point.

We are not here to confirm that more complex models always win. We are here to build the evaluation infrastructure that lets you find out — and defend the answer.

---

> **Instructor note — transition:** "Let's move to Module 2. We are going to lock the config and define the evaluation logic before we touch any data."
>
> Open `instructor_notebooks/02_framing_and_config.ipynb`.